<a href="https://colab.research.google.com/github/bhu27/Multimodal_Phishing_XAI/blob/main/modeltraning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from scipy.sparse import hstack
import joblib


In [3]:
df = pd.read_csv("sms_dataset_cleaned.csv")

print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nFirst 5 Records:")
display(df.head())

print("\nLabel Distribution:")
print(df["LABEL"].value_counts())

Dataset Shape: (10191, 5)

Column Names:
['LABEL', 'TEXT', 'URL', 'EMAIL', 'PHONE']

First 5 Records:


,LABEL,TEXT,URL,EMAIL,PHONE
0,ham,Hi! You just spoke to DEEPAK. We'd like to kno...,No,No,No
1,ham,Yay can't wait to party together!,No,No,No
2,ham,At what time are you coming.,No,No,No
3,smishing,"Dear customer your PAY2TMKYC has been expired,...",No,No,Yes
4,ham,Yo you around? A friend of mine's lookin to pi...,No,No,No



Label Distribution:
LABEL
ham         3397
smishing    3397
spam        3397
Name: count, dtype: int64


In [4]:
# Keep only the required columns
required = ["LABEL", "TEXT"]

for col in required:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

# Clean missing text
df["TEXT"] = df["TEXT"].fillna("").astype(str)

# Convert labels into two classes
df["LABEL"] = df["LABEL"].astype(str).str.strip().str.lower()

label_mapping = {
    "ham": "Legitimate",
    "legitimate": "Legitimate",
    "benign": "Legitimate",
    "normal": "Legitimate",
    "spam": "Phishing",
    "smishing": "Phishing",
    "phishing": "Phishing",
    "malicious": "Phishing"
}

df["LABEL"] = df["LABEL"].map(label_mapping)

# Remove rows with unknown labels
df = df.dropna(subset=["LABEL"])

print("Cleaned Dataset:", df.shape)
print(df["LABEL"].value_counts())

Cleaned Dataset: (10191, 5)
LABEL
Phishing      6794
Legitimate    3397
Name: count, dtype: int64


In [5]:
# Separate input and output
X = df[["TEXT"]].copy()
y = df["LABEL"]

# Convert URL, EMAIL and PHONE into binary features
for col in ["URL", "EMAIL", "PHONE"]:
    if col not in df.columns:
        df[col] = 0

    X[col] = (
        df[col].astype(str).str.strip().str.lower()
        .map({
            "yes": 1, "no": 0,
            "true": 1, "false": 0,
            "1": 1, "0": 0
        })
        .fillna(0)
    )

print("Feature preparation completed!")

Feature preparation completed!


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))


Training records: 8152
Testing records: 2039


In [7]:
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    ngram_range=(1, 2)
)

X_train_text = tfidf.fit_transform(X_train["TEXT"])

X_test_text = tfidf.transform(X_test["TEXT"])

print("TF-IDF feature extraction completed!")

TF-IDF feature extraction completed!


In [8]:
X_train_final = hstack([
    X_train_text,
    X_train[["URL", "EMAIL", "PHONE"]].values
])

X_test_final = hstack([
    X_test_text,
    X_test[["URL", "EMAIL", "PHONE"]].values
])

print("All features combined!")
print("Training shape:", X_train_final.shape)
print("Testing shape:", X_test_final.shape)

All features combined!
Training shape: (8152, 5003)
Testing shape: (2039, 5003)


In [9]:
model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train_final, y_train)

print("MODEL TRAINING COMPLETED!")

MODEL TRAINING COMPLETED!


In [10]:
y_pred = model.predict(X_test_final)

accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", round(accuracy * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Model Accuracy: 99.22 %

Classification Report:
              precision    recall  f1-score   support

  Legitimate       0.98      0.99      0.99       680
    Phishing       1.00      0.99      0.99      1359

    accuracy                           0.99      2039
   macro avg       0.99      0.99      0.99      2039
weighted avg       0.99      0.99      0.99      2039


Confusion Matrix:
[[ 676    4]
 [  12 1347]]


In [15]:
from sklearn.ensemble import RandomForestClassifier

In [11]:
# Save the model and TF-IDF vectorizer
joblib.dump(model, "phishing_model.pkl")
joblib.dump(tfidf, "tfidf_vectorizer.pkl")

print("Model and vectorizer saved!")

Model and vectorizer saved!


In [13]:
from google.colab import files

files.download("phishing_model.pkl")
files.download("tfidf_vectorizer.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
message = ["Congratulations! You won a free prize. Claim now!"]

text_features = tfidf.transform(message)

# Example URL, EMAIL and PHONE indicators
extra_features = np.array([[1, 0, 0]])

final_features = hstack([
    text_features,
    extra_features
])

prediction = model.predict(final_features)

print("Prediction:", prediction[0])

Prediction: Phishing


In [16]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf_model.fit(X_train_final, y_train)

print("Random Forest Training Completed!")

Random Forest Training Completed!


In [17]:
rf_pred = rf_model.predict(X_test_final)

print("Random Forest Accuracy:",
      round(accuracy_score(y_test, rf_pred) * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(y_test, rf_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rf_pred))

Random Forest Accuracy: 99.8 %

Classification Report:
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00       680
    Phishing       1.00      1.00      1.00      1359

    accuracy                           1.00      2039
   macro avg       1.00      1.00      1.00      2039
weighted avg       1.00      1.00      1.00      2039


Confusion Matrix:
[[ 678    2]
 [   2 1357]]


In [18]:
from sklearn.svm import LinearSVC


In [19]:
svm_model = LinearSVC(
    class_weight="balanced",
    random_state=42,
    max_iter=2000
)

svm_model.fit(X_train_final, y_train)

print("SVM Training Completed!")

SVM Training Completed!


In [20]:
svm_pred = svm_model.predict(X_test_final)

print("SVM Accuracy:",
      round(accuracy_score(y_test, svm_pred) * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(y_test, svm_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, svm_pred))

SVM Accuracy: 99.75 %

Classification Report:
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00       680
    Phishing       1.00      1.00      1.00      1359

    accuracy                           1.00      2039
   macro avg       1.00      1.00      1.00      2039
weighted avg       1.00      1.00      1.00      2039


Confusion Matrix:
[[ 678    2]
 [   3 1356]]


In [21]:
train_messages = set(X_train["TEXT"].str.strip().str.lower())
test_messages = set(X_test["TEXT"].str.strip().str.lower())

overlap = train_messages.intersection(test_messages)

print("Duplicate messages across train and test:",
      len(overlap))

Duplicate messages across train and test: 465


In [22]:
from sklearn.metrics import classification_report

print("RANDOM FOREST")
print(classification_report(
    y_test, rf_pred, digits=4
))

print("SVM")
print(classification_report(
    y_test, svm_pred, digits=4
))

print("LOGISTIC REGRESSION")
print(classification_report(
    y_test, y_pred, digits=4
))

RANDOM FOREST
              precision    recall  f1-score   support

  Legitimate     0.9971    0.9971    0.9971       680
    Phishing     0.9985    0.9985    0.9985      1359

    accuracy                         0.9980      2039
   macro avg     0.9978    0.9978    0.9978      2039
weighted avg     0.9980    0.9980    0.9980      2039

SVM
              precision    recall  f1-score   support

  Legitimate     0.9956    0.9971    0.9963       680
    Phishing     0.9985    0.9978    0.9982      1359

    accuracy                         0.9975      2039
   macro avg     0.9971    0.9974    0.9972      2039
weighted avg     0.9975    0.9975    0.9975      2039

LOGISTIC REGRESSION
              precision    recall  f1-score   support

  Legitimate     0.9826    0.9941    0.9883       680
    Phishing     0.9970    0.9912    0.9941      1359

    accuracy                         0.9922      2039
   macro avg     0.9898    0.9926    0.9912      2039
weighted avg     0.9922    0.9922   

In [23]:
# Create normalized text for duplicate detection
df["TEXT_CLEAN"] = (
    df["TEXT"].fillna("").astype(str)
    .str.strip().str.lower()
)

# Check messages having conflicting labels
label_counts = df.groupby("TEXT_CLEAN")["LABEL"].nunique()

conflicts = label_counts[label_counts > 1]

print("Messages with conflicting labels:", len(conflicts))

# Remove messages with conflicting labels
df_clean = df[~df["TEXT_CLEAN"].isin(conflicts.index)].copy()

# Remove duplicate messages
df_clean = df_clean.drop_duplicates(
    subset=["TEXT_CLEAN"],
    keep="first"
)

print("Original records:", len(df))
print("Clean unique records:", len(df_clean))

Messages with conflicting labels: 0
Original records: 10191
Clean unique records: 7439


In [24]:
X = df_clean[["TEXT"]].copy()
y = df_clean["LABEL"]

for col in ["URL", "EMAIL", "PHONE"]:
    X[col] = df_clean[col].map({
        "yes": 1, "no": 0,
        "true": 1, "false": 0,
        "1": 1, "0": 0
    }).fillna(0)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

Training records: 5951
Testing records: 1488


In [25]:
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    ngram_range=(1, 2)
)

X_train_text = tfidf.fit_transform(X_train["TEXT"])
X_test_text = tfidf.transform(X_test["TEXT"])

X_train_final = hstack([
    X_train_text,
    X_train[["URL", "EMAIL", "PHONE"]].values
])

X_test_final = hstack([
    X_test_text,
    X_test[["URL", "EMAIL", "PHONE"]].values
])

print("Feature extraction completed!")

Feature extraction completed!


In [26]:
# Logistic Regression
lr_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)
lr_model.fit(X_train_final, y_train)

# Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)
rf_model.fit(X_train_final, y_train)

# SVM
svm_model = LinearSVC(
    class_weight="balanced",
    random_state=42,
    max_iter=2000
)
svm_model.fit(X_train_final, y_train)

print("All three models retrained successfully!")

All three models retrained successfully!


In [27]:
models = {
    "Logistic Regression": lr_model,
    "Random Forest": rf_model,
    "SVM": svm_model
}

for name, model in models.items():
    predictions = model.predict(X_test_final)

    print("\n", name)
    print("Accuracy:",
          round(accuracy_score(y_test, predictions) * 100, 2), "%")

    print(classification_report(
        y_test, predictions, digits=4
    ))


 Logistic Regression
Accuracy: 98.79 %
              precision    recall  f1-score   support

  Legitimate     0.9882    0.9853    0.9867       679
    Phishing     0.9877    0.9901    0.9889       809

    accuracy                         0.9879      1488
   macro avg     0.9879    0.9877    0.9878      1488
weighted avg     0.9879    0.9879    0.9879      1488


 Random Forest
Accuracy: 99.19 %
              precision    recall  f1-score   support

  Legitimate     0.9840    0.9985    0.9912       679
    Phishing     0.9987    0.9864    0.9925       809

    accuracy                         0.9919      1488
   macro avg     0.9914    0.9925    0.9919      1488
weighted avg     0.9920    0.9919    0.9919      1488


 SVM
Accuracy: 99.26 %
              precision    recall  f1-score   support

  Legitimate     0.9912    0.9926    0.9919       679
    Phishing     0.9938    0.9926    0.9932       809

    accuracy                         0.9926      1488
   macro avg     0.9925    0.9

In [28]:
import joblib

joblib.dump(svm_model, "SMS_Phishing_SVM.pkl")
joblib.dump(tfidf, "SMS_TFIDF.pkl")

print("Final SMS model saved successfully!")

Final SMS model saved successfully!


In [29]:
from google.colab import files

files.download("SMS_Phishing_SVM.pkl")
files.download("SMS_TFIDF.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [32]:
def predict_sms(message):
    text_features = tfidf.transform([message])

    # Keep the same feature order used during training
    extra_features = [[
        int("http" in message.lower() or "www." in message.lower()),
        0,  # EMAIL indicator
        0   # PHONE indicator
    ]]

    import scipy.sparse as sp
    final_features = sp.hstack(
        [text_features, sp.csr_matrix(extra_features)]
    )

    prediction = svm_model.predict(final_features)[0]
    print("Prediction:", prediction)

In [33]:
predict_sms("Congratulations! You won a prize. Click this link to claim now.")
predict_sms("Hey, are we meeting for lunch today?")
predict_sms("Your bank account is blocked. Verify your details immediately.")

Prediction: Phishing
Prediction: Legitimate
Prediction: Phishing


In [34]:
predict_sms("Your OTP is 482913. Do not share it with anyone.")
predict_sms("Your bank KYC has expired. Update now: http://fakebank.com")
predict_sms("Mom, I will reach home in 10 minutes.")
predict_sms("Your account received Rs.5000. Click here to claim reward.")
predict_sms("Your electricity bill is due tomorrow. Please pay on time.")

Prediction: Legitimate
Prediction: Phishing
Prediction: Legitimate
Prediction: Phishing
Prediction: Legitimate


In [37]:
from sklearn.metrics import confusion_matrix

# Use the preprocessed features X_test_final instead of the raw X_test
print(confusion_matrix(y_test, svm_model.predict(X_test_final)))

[[674   5]
 [  6 803]]


In [39]:
from google.colab import files

files.download("SMS_Phishing_SVM.pkl")
files.download("SMS_TFIDF.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [38]:
import joblib

joblib.dump(svm_model, "SMS_Phishing_SVM.pkl")
joblib.dump(tfidf, "SMS_TFIDF.pkl")

print("Models saved successfully!")

Models saved successfully!


In [35]:
predict_sms("send me ur otp da")


Prediction: Legitimate


In [40]:
from google.colab import files
uploaded = files.upload()

Saving sms+spam+collection.zip to sms+spam+collection (1).zip


In [41]:
import zipfile

with zipfile.ZipFile("sms+spam+collection.zip", "r") as zip_ref:
    zip_ref.extractall("/content/sms_dataset")

print("Dataset extracted successfully!")

Dataset extracted successfully!


In [42]:
import os

print(os.listdir("/content/sms_dataset"))

['SMSSpamCollection', 'readme']


In [43]:
import pandas as pd

new_df = pd.read_csv(
    "/content/sms_dataset/SMSSpamCollection",
    sep="\t",
    header=None,
    names=["LABEL", "TEXT"],
    encoding="latin-1"
)

print(new_df.head(10))
print("\nDataset shape:", new_df.shape)
print("\nLabel distribution:")
print(new_df["LABEL"].value_counts())

  LABEL                                               TEXT
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...
5  spam  FreeMsg Hey there darling it's been 3 week's n...
6   ham  Even my brother is not like to speak with me. ...
7   ham  As per your request 'Melle Melle (Oru Minnamin...
8  spam  WINNER!! As a valued network customer you have...
9  spam  Had your mobile 11 months or more? U R entitle...

Dataset shape: (5572, 2)

Label distribution:
LABEL
ham     4825
spam     747
Name: count, dtype: int64


In [44]:
print("SVM expects features:", svm_model.n_features_in_)

print("TF-IDF vocabulary size:",
      len(tfidf.get_feature_names_out()))

print("Extra features:",
      svm_model.n_features_in_ -
      len(tfidf.get_feature_names_out()))

SVM expects features: 5003
TF-IDF vocabulary size: 5000
Extra features: 3


In [45]:
import numpy as np
import scipy.sparse as sp

# Convert labels
new_df["LABEL"] = new_df["LABEL"].map({
    "ham": "Legitimate",
    "spam": "Phishing"
})

# Extract SMS text
new_text = new_df["TEXT"].fillna("").astype(str)

# TF-IDF transformation using the ORIGINAL vectorizer
text_features = tfidf.transform(new_text)

# Extract URL, EMAIL and PHONE indicators
url_features = new_text.str.contains(
    r"http|www\.|https", case=False, regex=True
).astype(int)

email_features = new_text.str.contains(
    r"\b[\w.-]+@[\w.-]+\.\w+\b", regex=True
).astype(int)

phone_features = new_text.str.contains(
    r"\b\d{10,}\b", regex=True
).astype(int)

extra_features = np.column_stack([
    url_features,
    email_features,
    phone_features
])

# Combine all features
new_X = sp.hstack([
    text_features,
    sp.csr_matrix(extra_features)
])

new_y = new_df["LABEL"]

print("New dataset prepared!")
print("Feature shape:", new_X.shape)
print("Label shape:", new_y.shape)

New dataset prepared!
Feature shape: (5572, 5003)
Label shape: (5572,)


In [46]:
# Use the already-trained SVM model
new_predictions = svm_model.predict(new_X)

print("Predictions completed!")
print("Total predictions:", len(new_predictions))

Predictions completed!
Total predictions: 5572


In [47]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

print("Accuracy:",
      accuracy_score(new_y, new_predictions) * 100, "%")

print("\nClassification Report:\n")
print(classification_report(new_y, new_predictions))

print("\nConfusion Matrix:\n")
print(confusion_matrix(new_y, new_predictions))

Accuracy: 99.31801866475233 %

Classification Report:

              precision    recall  f1-score   support

  Legitimate       0.99      1.00      1.00      4825
    Phishing       0.98      0.96      0.97       747

    accuracy                           0.99      5572
   macro avg       0.99      0.98      0.99      5572
weighted avg       0.99      0.99      0.99      5572


Confusion Matrix:

[[4814   11]
 [  27  720]]


In [48]:
import re

def normalize_sms(text):
    return re.sub(r"\s+", " ", str(text).lower()).strip()

original_messages = set(
    df_clean["TEXT"].apply(normalize_sms)
)

external_messages = set(
    new_df["TEXT"].apply(normalize_sms)
)

overlap = original_messages.intersection(external_messages)

print("Matching messages:", len(overlap))
print("External dataset messages:", len(external_messages))
print("Unique external messages:",
      len(external_messages) - len(overlap))

Matching messages: 3169
External dataset messages: 5157
Unique external messages: 1988


In [49]:
# Identify unique external messages
new_df["NORMALIZED"] = new_df["TEXT"].apply(normalize_sms)

unique_df = new_df[
    ~new_df["NORMALIZED"].isin(original_messages)
].copy()

# Transform using original TF-IDF
unique_text = unique_df["TEXT"].fillna("").astype(str)

unique_tfidf = tfidf.transform(unique_text)

url = unique_text.str.contains(
    r"http|www\.|https", case=False
).astype(int)

email = unique_text.str.contains(
    r"\b[\w.-]+@[\w.-]+\.\w+\b"
).astype(int)

phone = unique_text.str.contains(
    r"\b\d{10,}\b"
).astype(int)

unique_extra = np.column_stack([url, email, phone])

unique_X = sp.hstack([
    unique_tfidf,
    sp.csr_matrix(unique_extra)
])

unique_y = unique_df["LABEL"]

# Predict without retraining
unique_predictions = svm_model.predict(unique_X)

print("Unique external messages:", len(unique_df))

print("\nAccuracy:")
print(accuracy_score(unique_y, unique_predictions) * 100)

print("\nClassification Report:")
print(classification_report(unique_y, unique_predictions))

print("\nConfusion Matrix:")
print(confusion_matrix(unique_y, unique_predictions))

Unique external messages: 2212

Accuracy:
98.64376130198916

Classification Report:
              precision    recall  f1-score   support

  Legitimate       0.99      1.00      0.99      1782
    Phishing       0.99      0.94      0.96       430

    accuracy                           0.99      2212
   macro avg       0.99      0.97      0.98      2212
weighted avg       0.99      0.99      0.99      2212


Confusion Matrix:
[[1776    6]
 [  24  406]]


In [50]:
!pip install xgboost

In [51]:
import xgboost as xgb
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("XGBoost version:", xgb.__version__)
print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

XGBoost version: 3.4.1
Training data shape: (5951, 4)
Testing data shape: (1488, 4)


In [53]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# Convert labels into numbers
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

# Create XGBoost model
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

# Train model using the numeric preprocessed features
xgb_model.fit(X_train_final, y_train_enc)

print("XGBoost training completed!")

XGBoost training completed!


In [55]:
# Make predictions using the preprocessed numerical feature matrix
xgb_predictions_enc = xgb_model.predict(X_test_final)

# Convert predictions back to original labels
xgb_predictions = le.inverse_transform(xgb_predictions_enc.astype(int))

# Accuracy
print("XGBoost Accuracy:", accuracy_score(y_test, xgb_predictions) * 100, "%")

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, xgb_predictions))

# Confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, xgb_predictions, labels=["Legitimate", "Phishing"]))

XGBoost Accuracy: 97.24462365591397 %

Classification Report:
              precision    recall  f1-score   support

  Legitimate       0.95      0.99      0.97       679
    Phishing       0.99      0.96      0.97       809

    accuracy                           0.97      1488
   macro avg       0.97      0.97      0.97      1488
weighted avg       0.97      0.97      0.97      1488


Confusion Matrix:
[[671   8]
 [ 33 776]]
